# PARC2026 予選配布リポジトリ セットアップ・提出用ノートブック（Colab版）

公式配布リポジトリ [matsuolab/PARC2026_pre](https://github.com/matsuolab/PARC2026_pre) を
Colab上でセットアップし、SmolVLAモデルをデプロイして提出用zipを作成・検証するまでを行う。

**このノートブックが担当する範囲**: 環境構築 → 学習済みモデルのデプロイ → 動作確認 →
提出用zip作成・検証。**SmolVLAのLoRA学習自体はこのノートブックの範囲外**
（別ノートブック`examples/smolvla_libero_spatial_lora.ipynb`で実施済みという前提）。

**前提条件**:
- Google Driveの`MyDrive/PARC2026/`に、学習済みモデル
  (`smolvla_libero_plus_spatial_lora_merged.zip`または展開済みフォルダ)が
  アップロード済みであること（初回のみ手動アップロードが必要、詳細はセクション6参照）
- **ランタイム > ランタイムのタイプを変更 > GPU** に設定してから実行すること

**ランタイムがリセットされた場合**: 学習をやり直す必要はない。このノートブックを
**上から順に実行し直すだけで復旧できる**ように設計されている（末尾の「再開方法」も参照）。

**方針(2026-08-04)**: Colab Pro(L4)を使用。SmolVLAは軽量モデルのため、
Pi0.5のようなOOM問題は起きにくい想定。詰まった場合はRunPodへ切り替える
（`docs/env_setup.md`にPi0.5時代の経緯を記録済み）。

## 0. GPU確認

In [1]:
!nvidia-smi

Tue Aug  4 01:59:42 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   49C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 0.5 MPLBACKEND環境変数の恒久修正（このセルは必ずGPU確認の直後・最初に1回だけ実行）

ColabのPythonカーネルは`MPLBACKEND=module://matplotlib_inline.backend_inline`という、ノートブック内表示専用の特殊なバックエンドを起動時から持っている。この値は`%%bash`セルや`subprocess.Popen`など、このカーネルから起動するあらゆる子プロセスにそのまま引き継がれる。LIBERO-Plus経由でmatplotlibをimportする箇所（`env_wrapper.py`）がこれを認識できず、`ValueError: Key backend: ... is not a valid value`で毎回同じ場所（疎通確認・提出物検証・カメラ診断など、pythonを起動する複数のセル）で繰り返し落ちていた。

個別のセルに`export MPLBACKEND=Agg`を都度書き足す対症療法を繰り返す代わりに、**Pythonカーネル自体の`os.environ`をここで一度だけ書き換える**ことで、以降のセルすべて（`%%bash`・`!command`・`subprocess.Popen`のいずれも）に自動的に継承させる。

In [ ]:
import os
os.environ['MPLBACKEND'] = 'Agg'
print('MPLBACKEND set to:', os.environ['MPLBACKEND'])

## 1. Python 3.10 の確保

`setup.sh` は `python3.10` を明示的に要求する。Colabのデフォルトはバージョンが異なる場合があるため、
存在確認し、なければ deadsnakes PPA から導入する。

In [2]:
import subprocess

has_py310 = subprocess.run(['which', 'python3.10'], capture_output=True).returncode == 0
print('python3.10 found:', has_py310)

if not has_py310:
    print('Installing python3.10 via deadsnakes PPA...')

python3.10 found: True


In [11]:
%%bash
# python3.10-venv と MagickWand (ImageMagick) を確実にインストールします
export DEBIAN_FRONTEND=noninteractive
apt-get update -qq
apt-get install -y -qq software-properties-common
add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
apt-get update -qq
apt-get install -y -qq python3.10 python3.10-venv python3.10-distutils python3.10-dev libmagickwand-dev
python3.10 --version

Selecting previously unselected package fonts-droid-fallback.
(Reading database ... 122427 files and directories currently installed.)
Preparing to unpack .../00-fonts-droid-fallback_1%3a6.0.1r16-1.1build1_all.deb ...
Unpacking fonts-droid-fallback (1:6.0.1r16-1.1build1) ...
Selecting previously unselected package imagemagick-6-common.
Preparing to unpack .../01-imagemagick-6-common_8%3a6.9.11.60+dfsg-1.3ubuntu0.22.04.5_all.deb ...
Unpacking imagemagick-6-common (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5) ...
Selecting previously unselected package libmagickcore-6-headers.
Preparing to unpack .../02-libmagickcore-6-headers_8%3a6.9.11.60+dfsg-1.3ubuntu0.22.04.5_all.deb ...
Unpacking libmagickcore-6-headers (8:6.9.11.60+dfsg-1.3ubuntu0.22.04.5) ...
Selecting previously unselected package libmagickcore-6-arch-config:amd64.
Preparing to unpack .../03-libmagickcore-6-arch-config_8%3a6.9.11.60+dfsg-1.3ubuntu0.22.04.5_amd64.deb ...
Unpacking libmagickcore-6-arch-config:amd64 (8:6.9.11.60+dfsg-1.3ub

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


## 2. 配布リポジトリをclone

In [4]:
import os

REPO_DIR = '/content/PARC2026_pre'
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/matsuolab/PARC2026_pre.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

Cloning into '/content/PARC2026_pre'...
remote: Enumerating objects: 43, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (20/20), done.
remote: Total 43 (delta 2), reused 3 (delta 2), pack-reused 21 (from 1)
Receiving objects: 100% (43/43), 73.02 KiB | 12.17 MiB/s, done.
Resolving deltas: 100% (2/2), done.


## 3. `setup.sh` 実行（初回のみ、アセット取得含め10〜20分程度）

venv作成・依存インストール(CPU torch)・LIBERO-plus/LIBEROの取得とパッチ・アセットダウンロード・
`~/.libero/config.yaml` 生成を一括で行う。

In [12]:
%%bash
# 環境変数の競合を避けつつ再セットアップします
cd /content/PARC2026_pre
# 【2026-08-05修正】無条件rm -rfは同一セッション内の再実行のたびに
# venv構築・pip install(torch/mujoco/robosuite等)をフルにやり直す原因になっていた。
# 壊れている(activateスクリプトが無い)場合だけ削除する、元々の安全な条件分岐に戻す。
if [ -d venv ] && [ ! -f venv/bin/activate ]; then
    echo "壊れたvenvディレクトリを検出、削除して再作成します"
    rm -rf venv
fi
export MPLBACKEND=Agg
bash setup.sh

[setup] 1/5 venv + 依存
[setup] 2/5 LIBERO-plus / LIBERO の取得
[setup] 3/5 LIBERO-plus パッチ
[setup] 4/5 アセット
[setup]   textures=583
[setup] 5/5 libero 設定
[setup]   既存の ~/.libero/config.yaml を config.yaml.bak に退避しました
[setup] 動作確認（suite 登録）
suite 登録 OK

セットアップ完了。評価を回すシェルで次を実行してください:
  source env.sh


[robosuite WARNING] No private macro file found! (__init__.py:7)
[robosuite WARNING] It is recommended to use a private macro file (__init__.py:8)
[robosuite WARNING] To setup, run: python /content/PARC2026_pre/venv/lib/python3.10/site-packages/robosuite/scripts/setup_macros.py (__init__.py:9)
Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


## 4. 疎通確認（ランダムポリシーで評価パイプラインを回す）

ここまでは学習不要。配布されたテンプレートのまま、Track1のexampleタスクで
評価パイプラインが最後まで動くことを確認する。

ポリシーサーバーをバックグラウンドで起動し、`pipeline` から接続する。

In [ ]:
# %%bash --bg はColab上で不安定なことがあるため、subprocess.Popenで確実にバックグラウンド起動する
import subprocess

log_file = open('/content/policy_server.log', 'w')
policy_proc = subprocess.Popen(
    ['/content/PARC2026_pre/venv/bin/python', 'submission_template/policy_server.py', '--port', '8000'],
    cwd='/content/PARC2026_pre',
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print('policy_server PID:', policy_proc.pid)

In [ ]:
import time

# プロセスが生きているか(poll()がNoneなら生存中、数値なら既にエラー終了している)を確認しながら待つ
for i in range(15):
    exit_code = policy_proc.poll()
    if exit_code is not None:
        print(f'policy_server は既に終了しています(exit code={exit_code})。ログを確認してください。')
        break
    time.sleep(2)
else:
    print('15回チェックしてもプロセスは生存中(異常終了なし)。/health を確認します。')

!curl -i http://localhost:8000/health

In [ ]:
%%bash
cd /content/PARC2026_pre
source env.sh
python -m pipeline --server-url http://localhost:8000 --track track1 --n-episodes 2 --max-steps 600

## 5. ログ確認（エラーが出た場合）

In [14]:
# サーバーログの末尾を表示してエラーがないか確認します
!tail -n 20 /content/policy_server.log

tail: cannot open '/content/policy_server.log' for reading: No such file or directory


## 6. Google Driveをマウントし、学習済みモデルを復元する

学習ノートブック側で出力した`smolvla_libero_plus_spatial_lora_merged.zip`（またはフォルダ）を
事前に`MyDrive/PARC2026/`にアップロードしておけば、Colabセッションを新しく開いてもここから読み込める。
zipのまま置いても、展開済みフォルダのまま置いても、どちらでも動くようにしている。

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

DRIVE_DIR = Path('/content/drive/MyDrive/PARC2026')
ZIP_PATH = DRIVE_DIR / 'smolvla_libero_plus_spatial_lora_merged.zip'
FOLDER_PATH = DRIVE_DIR / 'smolvla_libero_plus_spatial_lora_merged'

if FOLDER_PATH.exists():
    # 既に展開済みフォルダとして置かれている場合はそのまま使う
    DRIVE_MODEL_DIR = str(FOLDER_PATH)
    print('展開済みフォルダを検出:', DRIVE_MODEL_DIR)
elif ZIP_PATH.exists():
    # zipのまま置かれている場合は/content/へ展開してから使う
    # (Drive上で直接展開するとファイルI/Oが遅いため、一度ローカルディスクへ展開する)
    !unzip -oq "{ZIP_PATH}" -d /content/
    DRIVE_MODEL_DIR = '/content/smolvla_libero_plus_spatial_lora_merged'
    print('zipを展開:', DRIVE_MODEL_DIR)
else:
    raise FileNotFoundError(
        f'{FOLDER_PATH} も {ZIP_PATH} も見つかりません。'
        f'学習ノートブックの出力を MyDrive/PARC2026/ へアップロードしてから再実行してください。'
    )

print('中身:', __import__('os').listdir(DRIVE_MODEL_DIR))

## 7. 学習済みモデルを `model_weights/` へ配置

In [ ]:
import shutil
from pathlib import Path

MODEL_WEIGHTS_DIR = Path('/content/PARC2026_pre/submission_template/model_weights')
MODEL_WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(DRIVE_MODEL_DIR, MODEL_WEIGHTS_DIR, dirs_exist_ok=True)
print('モデル配置完了:', list(MODEL_WEIGHTS_DIR.iterdir()))

# 【2026-08-04追加】SmolVLM2のconfig/tokenizer(重み以外の軽量ファイル)をmodel_weights/vlm/へ同梱する。
# 評価中は外部ネットワークアクセスが禁止されており、SmolVLAPolicyはvlm_model_name(HF Hub上の文字列参照)
# からAutoConfig/AutoProcessorを必ず呼ぶため、ローカル同梱しないと採点環境でサーバー起動が失敗する。
VLM_DIR = MODEL_WEIGHTS_DIR / 'vlm'
VLM_DIR.mkdir(parents=True, exist_ok=True)
!curl -sL -o {VLM_DIR}/added_tokens.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/added_tokens.json
!curl -sL -o {VLM_DIR}/chat_template.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/chat_template.json
!curl -sL -o {VLM_DIR}/config.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/config.json
!curl -sL -o {VLM_DIR}/generation_config.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/generation_config.json
!curl -sL -o {VLM_DIR}/merges.txt https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/merges.txt
!curl -sL -o {VLM_DIR}/preprocessor_config.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/preprocessor_config.json
!curl -sL -o {VLM_DIR}/processor_config.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/processor_config.json
!curl -sL -o {VLM_DIR}/special_tokens_map.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/special_tokens_map.json
!curl -sL -o {VLM_DIR}/tokenizer.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/tokenizer.json
!curl -sL -o {VLM_DIR}/vocab.json https://huggingface.co/HuggingFaceTB/SmolVLM2-500M-Video-Instruct/resolve/main/vocab.json
print('VLM assets配置完了:', list(VLM_DIR.iterdir()))

## 7.5【最重要】ネットワーク遮断下での起動確認

**評価中は外部ネットワークアクセスが禁止されている**。ローカルColabでの動作確認は
学習時にHFキャッシュが温まっているため、この問題を原理的に検出できない
（2026-08-04 planner設計で判明）。`HF_HUB_OFFLINE=1`かつキャッシュなしの状態で、
本番同様の条件を再現してサーバーが起動するか確認する。

In [ ]:
import subprocess
import time
import os

# 既存サーバーが動いていれば止める
if 'policy_proc' in globals() and policy_proc.poll() is None:
    policy_proc.terminate()
    policy_proc.wait(timeout=10)

# キャッシュなし・オフライン強制の環境変数でサーバーを起動する
# 【重要】venvのpythonを明示的に指定する。素の'python'だと別の環境(学習ノートブック側で
# 別途lerobotが入ったシステムPython等)を見てしまい、venv+requirements.txtで実際に
# インストールされる内容と異なる偽陽性のテストになる恐れがある(2026-08-05レビューで発見)。
env = os.environ.copy()
env['HF_HOME'] = '/content/empty_hf_cache_for_offline_test'
env['HF_HUB_OFFLINE'] = '1'
env['TRANSFORMERS_OFFLINE'] = '1'
os.makedirs(env['HF_HOME'], exist_ok=True)

log_file = open('/content/policy_server_offline_test.log', 'w')
offline_test_proc = subprocess.Popen(
    ['/content/PARC2026_pre/venv/bin/python', 'submission_template/policy_server.py', '--port', '8001'],
    cwd='/content/PARC2026_pre',
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env=env,
)
print('offline test PID:', offline_test_proc.pid)

for i in range(60):
    exit_code = offline_test_proc.poll()
    if exit_code is not None:
        print(f'サーバーが終了しました(exit code={exit_code})。ネットワーク遮断下で起動できていません。ログを確認してください。')
        break
    time.sleep(2)
else:
    print('60回チェックしても生存中。オフラインでも起動できている可能性が高い。/healthを確認します。')

!curl -i http://localhost:8001/health

In [ ]:
# 失敗した場合はここでログを確認する(不足しているファイル名がエラーに出るはず)
!tail -n 60 /content/policy_server_offline_test.log

# テスト用サーバーを停止
if offline_test_proc.poll() is None:
    offline_test_proc.terminate()

## 8. `MyPolicy`(SmolVLA)を自分のリポジトリから取得して反映

手動コピペではなく、検証済みの`policy_server.py`を`norikioka/parc2026`から直接取得して上書きする。
これにより、貼り付けミス(インデント崩れ・二重定義など)が起きなくなる。
コード側を修正した場合は、ローカルでpushしてからこのセルを再実行すれば反映される。

In [ ]:
!curl -s -o /content/PARC2026_pre/submission_template/policy_server.py \
  https://raw.githubusercontent.com/norikioka/parc2026/master/src/parc2026/policy_server_smolvla_full.py

!curl -s -o /content/PARC2026_pre/submission_template/requirements.txt \
  https://raw.githubusercontent.com/norikioka/parc2026/master/src/parc2026/requirements_smolvla.txt

# 構文チェック(ここでエラーが出たら取得内容がおかしいので、ダウンロード元URLを確認する)
!python3 -c "import ast; ast.parse(open('/content/PARC2026_pre/submission_template/policy_server.py').read()); print('syntax OK')"

# requirements.txtをvenvへインストール(実際に提出されるものと同じ内容・同じ環境で検証するため)
# 【重要】git+https://形式はOmnicampus提出物バリデーションで禁止と判明したため、
# requirements_smolvla.txt側は既にPyPI版(lerobot[smolvla]==0.6.0)に修正済み(2026-08-04)。
# ここでは絶対にgit+https経由のインストールを行わないこと。
!/content/PARC2026_pre/venv/bin/pip install -q -r /content/PARC2026_pre/submission_template/requirements.txt
# 【2026-08-05修正】setup.shはCPU版torchを入れる仕様のため、後で手動でGPU版に
# 差し替えていた場合でもrequirements.txt再インストールで静かにCPU版へ巻き戻るリスクがある。
# 10秒/ステップ制約でCPU推論は致命傷になりうるため、print(見逃し可能)ではなくassert(即失敗)にする。
!/content/PARC2026_pre/venv/bin/python -c "import torch; assert torch.cuda.is_available(), 'GPU torchではありません。CPU版に巻き戻っている可能性があります'; print('CUDA available: True')"

## 9. サーバーを再起動してTrack1で疎通確認(SmolVLAモデル使用)

既存のサーバープロセス(セクション4でランダムポリシーで起動したもの)が生きていれば停止し、
更新した`policy_server.py`(SmolVLA実装入り)で起動し直す。

In [ ]:
import subprocess
import time

if 'policy_proc' in globals() and policy_proc.poll() is None:
    policy_proc.terminate()
    policy_proc.wait(timeout=10)
    print('既存プロセスを停止しました')

log_file = open('/content/policy_server.log', 'w')
policy_proc = subprocess.Popen(
    ['/content/PARC2026_pre/venv/bin/python', 'submission_template/policy_server.py', '--port', '8000'],
    cwd='/content/PARC2026_pre',
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
print('policy_server PID:', policy_proc.pid)

# モデルロード(SmolVLAPolicy.from_pretrained等)が走るため、ランダムポリシーより起動に時間がかかる想定
for i in range(60):
    exit_code = policy_proc.poll()
    if exit_code is not None:
        print(f'既に終了しています(exit code={exit_code})。ログを確認してください。')
        break
    time.sleep(2)
else:
    print('生存中。/health を確認します。')

!curl -i http://localhost:8000/health

In [ ]:
%%bash
cd /content/PARC2026_pre
source env.sh
python -m pipeline --server-url http://localhost:8000 --track track1 --n-episodes 2 --max-steps 600

## 10. 提出用zipを作成する

In [ ]:
import subprocess

submission_zip = '/content/PARC2026_pre/submission.zip'
result = subprocess.run(
    ['zip', '-r', submission_zip, 'policy_server.py', 'requirements.txt', 'model_weights/'],
    cwd='/content/PARC2026_pre/submission_template',
    capture_output=True, text=True,
)
print(result.stdout[-1500:])
print(result.stderr[-1500:])

!ls -lh {submission_zip}
!unzip -l {submission_zip}

## 11.【提出前に必ず実行】ローカルバリデーション

Omnicampusへの1日1回の提出を無駄にしないため、**ここで`PASS`が出るまでは絶対に提出しない**。

In [ ]:
%%bash
cd /content/PARC2026_pre
source env.sh
python validate_submission.py submission.zip

## 12.【参考・調査済み】診断: front/wristカメラ対応の実データ確認

**2026-08-04実施・結論確定済み: カメラ対応は正しいと確認済み**（front=俯瞰視点、
wrist=手先アップの画像を目視確認）。再実行は不要。再発防止のための記録として残している。

In [ ]:
%%bash
cd /content/PARC2026_pre
source env.sh
python3 -c "
from lerobot.datasets.lerobot_dataset import LeRobotDataset
import numpy as np
from PIL import Image

ds = LeRobotDataset('lerobot/libero_plus', revision='f3f49f426d75030177b18778374005bc12ccd588')
sample = ds[0]

for key in ['observation.images.front', 'observation.images.wrist']:
    img = sample[key]
    arr = (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8) if img.max() <= 1.0 else img.permute(1, 2, 0).numpy().astype(np.uint8)
    out_path = f'/content/camera_check_{key.split(\".\")[-1]}.png'
    Image.fromarray(arr).save(out_path)
    print(f'saved: {out_path} shape={arr.shape}')
"


In [ ]:
from IPython.display import Image as IPyImage, display

print('front (俯瞰視点であるはず):')
display(IPyImage('/content/camera_check_front.png'))
print('wrist (手元アップであるはず):')
display(IPyImage('/content/camera_check_wrist.png'))

## 完了後: Omnicampusへの提出

セクション11で`PASS`を確認できたら、`/content/PARC2026_pre/submission.zip`を
Omnicampusへアップロードする（1日1回の制限に注意、余裕を持って行う）。

## ランタイムがリセットされた場合の再開方法

このノートブックは「ランタイム切断→再接続」で全て消える前提で、上から順に再実行すれば
必ず復旧できるように作られている。**学習をやり直す必要は一切ない**（学習済みモデルは
セクション6でGoogle Driveから復元する）。上から順に実行するだけでよい。

## 再実行のコツ

- モデルやコードを変更するたびに新しいセルを積み増す必要はない。該当セクションのセルを
  そのまま再実行すればよい(Jupyterのセルは何度でも上書き実行できる)
- `policy_server.py`または`requirements.txt`をローカルで修正した場合は、
  `~/projects/PARC/src/parc2026/`内のファイルを編集してpushしてから、
  セクション8のセルを再実行すれば反映される(手動コピペ不要)
- 詰まった場合は`docs/env_setup.md`の「つまずきポイント一覧」を確認・追記する

## 次のステップ

提出後は`docs/strategy.md`の判断ゲート（公式参考スコア0.0633との比較）に従い、
ロバスト性対策（ステップ4）に進むか、基本動作の見直しに戻るかを判断する。